# Jupyter

**Domain:** AI/ML Tooling  ·  **runnable:** yes

A refresher on the Jupyter ecosystem: the notebook document format, the kernel
protocol that runs your code, and the IPython features (rich display, magics)
that make notebooks more than a REPL with scrollback.

## 1. What & Why

**Jupyter** is an interactive computing environment built around two pieces: a
**notebook** (a JSON document of ordered *cells* — code, Markdown, raw — interleaved
with their outputs) and a **kernel** (a language process that executes the code and
streams results back). The front-end (JupyterLab, classic Notebook, VS Code, Colab)
is just a client that talks to the kernel over a messaging protocol.

**The problem it solves.** It collapses the edit → run → inspect loop into one
surface. You run a cell, see the table/plot/traceback right under it, tweak, and
re-run — without re-loading data or restarting a script. State (variables, imported
modules, a trained model) persists in the kernel between cells, so expensive setup
happens once. The result is also a shareable artifact: code, prose, math, and figures
in a single file that renders on GitHub and converts to HTML/PDF/slides.

**Reach for it when** you're exploring data, prototyping a model, teaching, or writing
a computational narrative where seeing intermediate output matters. **Reach for a
plain `.py` module when** you're building something that must be tested, imported,
diffed, and deployed — notebooks resist all four (see *When to Use vs Alternatives*).

## 2. Mental Model

Think of a notebook as a **remote control pointed at a long-running Python process**.

```
  ┌─────────────┐   ZeroMQ messages    ┌──────────────┐
  │  Front-end  │  ───────────────────▶│    Kernel    │
  │ (Lab / VS   │   execute_request    │  (IPython =  │
  │  Code/Colab)│◀─────────────────────│  Python proc)│
  │             │   stream / result    │              │
  └─────────────┘                      └──────────────┘
        │                                     │
   reads/writes                          holds live state:
   the .ipynb JSON                       vars, imports, model
```

Two consequences fall out of this picture and explain almost every Jupyter surprise:

1. **The document and the kernel are separate things.** The cells you *see* are stored
   in the `.ipynb` file; the *state* lives in the kernel's memory. They drift apart the
   moment you run cells out of order or delete a cell whose variable is still defined.
2. **Cell order on screen ≠ execution order.** The kernel only knows the sequence you
   pressed *Run*, tracked by the `In [n]` counter. "Restart & Run All" is the only way
   to prove the document reproduces top-to-bottom.

## 3. Key Concepts

- **Kernel** — the process that executes cells (IPython for Python; others exist for R,
  Julia, etc.). Restarting it wipes all in-memory state. *Interrupt* sends `KeyboardInterrupt`.
- **Cell & execution count** — `In [n]` is a monotonic counter of *runs*, not line
  numbers. A high or out-of-order `n` is the tell-tale of hidden state.
- **`.ipynb`** — a JSON file: a list of cells, each with `source`, `cell_type`,
  `outputs`, and `metadata`, plus top-level `metadata` (kernelspec) and `nbformat`.
- **Rich display / `DisplayObject`** — the kernel can emit multiple MIME types per
  result (text, HTML, PNG, JSON). The front-end renders the richest one it understands.
  `_repr_html_` / `_repr_png_` on an object hook into this.
- **Magics** — IPython meta-commands. *Line magics* (`%timeit`, `%who`, `%matplotlib`)
  start with one `%`; *cell magics* (`%%time`, `%%writefile`, `%%bash`) start with `%%`
  and consume the whole cell.
- **`Out` / underscores** — the last expression's value is auto-displayed and stored in
  `Out[n]`; `_`, `__`, `___` hold the last three results.
- **JupyterLab vs Notebook vs server** — `jupyter lab` is the modern IDE-like front-end;
  `jupyter notebook` is the classic one; both connect to a `jupyter server`. **IPython**
  is the kernel/REPL underneath, usable without any notebook at all.

## 4. Setup

Install JupyterLab (pulls in the server, IPython kernel, and nbconvert):

```bash
pip install jupyterlab        # or: uv add jupyterlab / conda install -c conda-forge jupyterlab
jupyter lab                   # launches the browser front-end
```

Useful adjacent commands:

```bash
jupyter kernelspec list                       # which kernels are registered
python -m ipykernel install --user --name myenv  # expose a venv as a kernel
jupyter nbconvert --to notebook --execute nb.ipynb  # run headless (CI)
jupyter nbconvert --to html nb.ipynb          # export
```

The cell below runs inside an existing kernel, so it just reports versions rather than
re-installing. Uncomment the `%pip` line in a fresh environment.

In [1]:
# %pip install jupyterlab ipython
import sys, IPython
from IPython import get_ipython

print("Python :", sys.version.split()[0])
print("IPython:", IPython.__version__)
print("Kernel running?", get_ipython() is not None)  # True inside a notebook kernel

Python : 3.13.7
IPython: 9.14.1
Kernel running? True


## 5. Worked Examples

### Example 1 — Rich display: one result, many MIME types

A kernel doesn't just print strings. `display()` (and the auto-shown last expression)
ask the object for the richest representation it offers. Below we render Markdown and
HTML, then define a class with a `_repr_html_` hook so Jupyter shows it as a table.

In [2]:
from IPython.display import display, Markdown, HTML

display(Markdown("**Markdown** rendered from the kernel: $e^{i\\pi}+1=0$"))
display(HTML("<span style='color:teal'>HTML</span> works too"))

class Color:
    """Any object can opt into rich display via _repr_html_."""
    def __init__(self, hexcode):
        self.hex = hexcode
    def _repr_html_(self):
        return f"<div style='background:{self.hex};width:80px;height:20px'></div>{self.hex}"

Color("#4c9aff")  # last expression -> auto-displayed using the richest repr

**Markdown** rendered from the kernel: $e^{i\pi}+1=0$

### Example 2 — Magics and hidden state

Magics are the notebook's power tools. `%timeit` micro-benchmarks a line, `%who_ls`
lists user variables, and `Out` lets you reach back at any prior result. This also
demonstrates the *execution-order* trap: the variables visible here exist only because
the earlier cells ran first.

In [3]:
# Line magic: time a tiny computation (CPU-friendly, deterministic-ish)
result = %timeit -o -n 1000 -r 3 sum(range(100))
print(f"best: {result.best*1e9:.0f} ns per call")

# Introspect kernel state: which user variables are alive right now?
user_vars = %who_ls
print("live user variables:", user_vars)

# The 'Color' class from Example 1 is still in memory -> proves state persists across cells
print("Color still defined?", "Color" in user_vars)

472 ns ± 4.13 ns per loop (mean ± std. dev. of 3 runs, 1,000 loops each)
best: 466 ns per call
live user variables: ['Color', 'HTML', 'IPython', 'Markdown', 'display', 'get_ipython', 'result', 'sys']
Color still defined? True


### Example 3 — Optional: gate a network/credential cell

A reproducible notebook must still execute when secrets or big downloads are absent.
The pattern: guard the side-effecting call behind `os.getenv` so a fresh kernel runs
the cell either way and the *shape* of the call stays visible.

In [4]:
import os

token = os.getenv("HF_TOKEN")  # or OPENAI_API_KEY, etc.
if token:
    # Real network call would go here, e.g. download a dataset / hit an API.
    print("Token found - would run the live call now.")
else:
    print("No HF_TOKEN set - skipping the network cell (notebook still executes).")

No HF_TOKEN set - skipping the network cell (notebook still executes).


## 6. Gotchas & Pitfalls

- **Out-of-order execution / hidden state.** Re-running cells up and down builds a kernel
  state no top-to-bottom reader can reproduce. Before committing, do **Kernel → Restart &
  Run All**. A non-monotonic `In [n]` is the warning sign.
- **Outputs are stored in the file.** `.ipynb` saves cell *outputs* — including big base64
  PNGs and, worse, secrets you printed. Strip them in CI with
  [`nbstripout`](https://github.com/kynan/nbstripout) or `jupyter nbconvert --clear-output`.
- **Diffs and merges are painful.** The JSON + embedded outputs make `git diff` noisy and
  merge conflicts brutal. Use `nbstripout` as a git filter, or tools like
  [`jupytext`](https://github.com/mwouts/jupytext) / `nbdime`.
- **Wrong kernel.** The notebook's kernelspec may point at a different environment than the
  one where you `pip install`ed a package, giving phantom `ModuleNotFoundError`s. Check the
  kernel name in the top-right and `import sys; sys.executable`.
- **`!pip` vs `%pip`.** `!pip install` shells out and may target the wrong Python; `%pip`
  installs into the *kernel's* environment. Prefer `%pip` / `%conda`.
- **Long-running / infinite cells.** A runaway cell blocks the kernel; *Interrupt* may not
  catch C-extension loops — sometimes you must restart and lose state.
- **Notebooks aren't importable.** You can't `import analysis.ipynb` cleanly; factor reusable
  logic into a `.py` module the notebook imports.

## 7. When to Use vs Alternatives

| Situation | Reach for | Why |
|---|---|---|
| Data exploration, plotting, prototyping | **Jupyter / JupyterLab** | Live state + inline rich output is the whole point. |
| Teaching, reports, reproducible papers | **Jupyter** (+ nbconvert / Quarto) | Code + prose + figures in one shareable artifact. |
| Library / production code, tested & deployed | **`.py` modules + pytest** | Importable, diffable, CI-friendly; notebooks resist all three. |
| Quick throwaway REPL, no document needed | **IPython** (`ipython`) | Same kernel, no JSON file to manage. |
| Want notebooks but clean git diffs | **Jupytext** / **Marimo** | Store as plain `.py`/`.md`; Marimo is reactive & git-friendly. |
| Zero-install, GPU, sharing | **Google Colab** / hosted JupyterHub | Browser-only, managed kernels. |
| Dashboards from notebook code | **Voila**, **Panel**, **Streamlit** | Turn cells into an app without the editing UI. |

**Honest trade-offs.** Jupyter wins on *interactivity and presentation* and loses on
*software-engineering hygiene*: hidden state, hard diffs, no native testing/imports.
The common discipline is to explore in a notebook, then graduate stable code into a
module and keep the notebook as a thin driver. Reactive notebooks (Marimo) and paired
text files (Jupytext) try to close the hygiene gap without abandoning interactivity.

## 8. Resources

- **JupyterLab documentation** — https://jupyterlab.readthedocs.io/en/stable/
- **Jupyter project docs (architecture, ecosystem)** — https://docs.jupyter.org/en/latest/
- **IPython docs (magics, rich display, the kernel)** — https://ipython.readthedocs.io/en/stable/
- **`nbformat` spec (the .ipynb JSON schema)** — https://nbformat.readthedocs.io/en/latest/
- **nbconvert (headless execution & export)** — https://nbconvert.readthedocs.io/en/latest/
- **Jupytext (notebooks as plain text for git)** — https://jupytext.readthedocs.io/en/latest/